In [56]:
import os
import mlflow
import numpy as np
import pandas as pd
import mlflow.sklearn
from pathlib import Path
import matplotlib.pyplot as plt 
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier 
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer
from sklearn.metrics import roc_auc_score, precision_score, recall_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder

In [8]:
df = pd.read_csv(r'C:\Users\priya\Desktop\PyCh_Pro\Churn_Analysis_and_Modelling\data\raw\telco_churn.csv')

In [24]:
df = df.drop(columns = ["customerID"])

In [25]:
df.sample(5)

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
934,Female,0,No,No,12,No,No phone service,DSL,Yes,No,No,Yes,No,No,One year,Yes,Mailed check,33.60,435.45,No
5585,Male,0,No,No,1,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Month-to-month,Yes,Mailed check,19.30,19.3,Yes
1638,Female,0,Yes,Yes,68,Yes,Yes,Fiber optic,No,Yes,Yes,No,No,No,Two year,Yes,Credit card (automatic),84.40,5746.75,No
1763,Female,0,No,No,2,Yes,No,DSL,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,45.40,80.95,No
3381,Female,0,No,No,41,Yes,No,DSL,Yes,No,Yes,Yes,Yes,Yes,One year,Yes,Bank transfer (automatic),79.85,3320.75,No


## Started Preparing Data

In [40]:
from sklearn.preprocessing import FunctionTransformer

def preprocessing_raw_data(X):
    df = X.copy()
    internet_cols = [
    "OnlineSecurity", "OnlineBackup", "DeviceProtection",
    "TechSupport", "StreamingTV", "StreamingMovies"
    ]
    phone_cols = ["MultipleLines"]
    binary_cols = [
        "OnlineSecurity", "OnlineBackup", "DeviceProtection",
        "TechSupport", "StreamingTV", "StreamingMovies",
        "MultipleLines", "Partner", "Dependents",
        "PhoneService", "PaperlessBilling"
    ]

    
    for col in binary_cols:
        if col in df.columns:
            df[col] = df[col].astype(str).str.strip().str.lower()
            
    for col in internet_cols:
        if col in df.columns:
            df[col] = df[col].replace("no internet service", "no")
            
    for col in phone_cols:
        if col in df.columns:
            df[col] = df[col].replace("no phone service", "no")
            
    df["Stability"] = df["Partner"].astype(str) + "_" + df["Dependents"].astype(str)
    
    return df

new_feature_clean_transformer = FunctionTransformer(preprocessing_raw_data)

In [39]:
def binaryEncoder(X):
    df = X.copy()
    binary_cols = [
        "OnlineSecurity", "OnlineBackup", "DeviceProtection",
        "TechSupport", "StreamingTV", "StreamingMovies",
        "MultipleLines", "Partner", "Dependents",
        "PhoneService", "PaperlessBilling"
    ]
    
    mapping = {"no": 0, "yes": 1}
    
    for col in binary_cols:
        if col in df.columns:
            # map it, and used fillna just in case a weird value sneaks in
            df[col] = df[col].map(mapping).fillna(df[col])
            
    return df

binary_encoder_transformer = FunctionTransformer(binaryEncoder)

In [17]:
def precision_recall_at_k(y_test, y_prob, return_precision = True, k = 0.2):
    """
    Compute Recall@K and Precision@K for ranking-based evaluation.
    
    Parameters:
    - y_test: true labels (0/1)
    - y_prob: predicted probabilities
    - k: fraction of top samples to consider
    
    Returns:
    - recall_at_k, precision_at_k
    """
    df_eval = pd.DataFrame({
        "prob" : y_prob,
        "actual" : y_test.values
    })
    df_eval = df_eval.sort_values(by = "prob", ascending = False)

    top_k = max(1, int(k * len(df_eval)))
    top_churners = df_eval.head(top_k)
    
    total_churn = df_eval["actual"].sum()
    if total_churn == 0:
        return (0, 0) if return_precision else 0
    recall = top_churners["actual"].sum() / total_churn
    # out of all the churners, how many churns are catched in the top 20%
    if return_precision:
        precision = top_churners["actual"].mean() if len(top_churners) > 0 else 0
        # in the top 20% range, how many are actual churns.
        return recall, precision

    return recall

## Building pipeline for Classification

In [45]:
X = df.drop(columns = ["Churn"]) # shape(7043, 19)
y = df["Churn"] # shape(7043,)

y_mapped = df["Churn"].map({"Yes": 1, "No": 0})

# first split
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y_mapped, test_size=0.30, stratify=y, random_state=42
)

# Second split
X_test, X_stream, y_test, y_stream = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=42
)

print("Data Split Complete:")
print(f"Training data: {X_train.shape[0]} rows (70%)")
print(f"Testing data:  {X_test.shape[0]} rows (15%)")
print(f"Stream data:   {X_stream.shape[0]} rows (15%)")



Data Split Complete:
Training data: 4930 rows (70%)
Testing data:  1056 rows (15%)
Stream data:   1057 rows (15%)


In [34]:
df.columns

Index(['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure',
       'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity',
       'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV',
       'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod',
       'MonthlyCharges', 'TotalCharges', 'Churn'],
      dtype='object')

In [35]:
scale_trf = ColumnTransformer(
    transformers=[
        # Name of step, The Transformer, The columns to apply it to
        ("scaling", StandardScaler(), ["Tenure", "MonthlyCharges", "TotalCharges"])
    ],
    remainder="passthrough"
)

In [46]:
encode_trf = ColumnTransformer(
    transformers = [
        ("multiple_cat_col_ohe", OneHotEncoder(sparse_output=False, handle_unknown="ignore"), ["InternetService", "PaymentMethod", "gender"]),
        ("ordinal_encoder_contract", OrdinalEncoder(categories = [["Month-to-month", "One year", "Two year"]]), ["Contract"])
    ], 
    remainder = "passthrough"
)

In [54]:
model1 = RandomForestClassifier()
random_pipeline = Pipeline([
    ("new_feature_and_data_cleaning", new_feature_clean_transformer),
    ("binary_encoder", binary_encoder_transformer),
    ("Scaling_transformer", scale_trf),
    ("encoding_transformer", encode_trf),
    ("Model", model1)
])

In [55]:
model2 = XGBClassifier()
xgb_pipeine = Pipeline([
    ("new_feature_and_data_cleaning", new_feature_clean_transformer),
    ("binary_encoder", binary_encoder_transformer),
    ("Scaling_transformer", scale_trf),
    ("encoding_transformer", encode_trf),
    ("Model", model2)
])

In [ ]:
print("--- 1. Connecting to the MLflow Vault ---")
# Set up the strict master path and bypass database
master_path = Path("C:/Users/priya/Desktop/PyCh_Pro/Churn_Analysis_and_Modelling")
db_uri = f"sqlite:///{master_path.as_posix()}/mlflow_v2.db"
mlflow.set_tracking_uri(db_uri)

experiment_name = "Telecom_Churn_Final_Pipelines"
mlflow.set_experiment(experiment_name)
print(f"Tracking URI locked to: {mlflow.get_tracking_uri()}")
print(f"Active Experiment: '{experiment_name}'\n")


print("--- 2. Injecting Winning Parameters ---")
# 🚨 REPLACE THESE NUMBERS WITH YOUR ACTUAL WINNERS FROM THE UI 🚨

# Assuming your pipeline step is named 'classifier'
random_pipeline.set_params(
    classifier__n_estimators=300,
    classifier__max_depth=15,
    classifier__min_samples_leaf=2,
    classifier__min_samples_split=5,
    classifier__class_weight="balanced",
    classifier__random_state=42
)
print("✅ Random Forest parameters injected.")

# Assuming your pipeline step is named 'classifier'
xgb_pipeline.set_params(
    classifier__n_estimators=200,
    classifier__learning_rate=0.05,
    classifier__max_depth=5,
    classifier__subsample=0.8,
    classifier__scale_pos_weight=1.35, # Replace with your actual ratio
    classifier__eval_metric="logloss",
    classifier__random_state=42
)
print("✅ XGBoost parameters injected.\n")


print("--- 3. Starting the Final Pipeline Arena ---")
models_to_test = {
    "Production_RF_Pipeline": random_pipeline,
    "Production_XGB_Pipeline": xgb_pipeline
}

for model_name, model_pipeline in models_to_test.items():
    
    with mlflow.start_run(run_name=model_name):
        print(f"Training and Evaluating: {model_name}...")
        
        # 1. Fit the entire pipeline on RAW data
        model_pipeline.fit(X_train, y_train)
        
        # 2. Predict on RAW data
        y_prob = model_pipeline.predict_proba(X_test)[:, 1]
        y_pred = model_pipeline.predict(X_test)
        
        # 3. Calculate Metrics
        roc_auc = roc_auc_score(y_test, y_prob)
        precision = precision_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        recall_20, precision_20 = precision_recall_at_k(y_test, y_prob) 
        
        # 4. Log Metrics to MLflow
        mlflow.log_param("pipeline_type", model_name)
        mlflow.log_metric("roc_auc", roc_auc)
        mlflow.log_metric("precision", precision)
        mlflow.log_metric("recall", recall)
        mlflow.log_metric("recall_20", recall_20)
        mlflow.log_metric("precision_20", precision_20)
        
        # 5. Log the ENTIRE pipeline safely to MLflow
        mlflow.sklearn.log_model(model_pipeline, "model")
        
        print(f"✅ Logged to MLflow | ROC-AUC: {roc_auc:.4f} | Recall: {recall:.4f} | Recall@20: {recall_20:.4f}\n")

print("🏆 Final Pipeline Test Complete!")
print("Run 'mlflow ui --backend-store-uri sqlite:///mlflow_v2.db' to see the final saved pipelines.")